# 06.07_All_mtx_to_RDS_R

Matrix Market 转 Seurat RDS。

- 当前文件：`analysis/06_single_cell_analysis/06.07_All_mtx_to_RDS_R.ipynb`
- 原始来源：`Codes/06.07_R_mtx_to_RDS.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`Matrix`, `Seurat`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_base

## Auco 测试

In [ ]:
# 安装和加载需要的包
# install.packages("~/zz/Software/Rpackages/BiocManager_1.30.19.tar.gz", repos = NULL, type = "source")
# install.packages("~/zz/Software/Rpackages/igraph_1.3.5.tar.gz", repos = NULL, type = "source")
# install.packages("~/zz/Software/Rpackages/leiden_0.4.3.tar.gz", repos = NULL, type = "source")
# install.packages("~/zz/Software/Rpackages/spatstat.core_2.4-4.tar.gz", repos = NULL, type = "source")
# install.packages("~/zz/Software/Rpackages/Seurat_4.2.0.tar.gz", repos = NULL, type = "source")
# install.packages("~/zz/Software/Rpackages/SingleCellExperiment_1.24.0.tar.gz", repos = NULL, type = "source")

# 加载必要的库
library(Seurat)

if (!requireNamespace("Matrix", quietly = TRUE))
  install.packages("Matrix")
library(Matrix)

# Human
# 读取稀疏矩阵
expression_matrix <- readMM("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.matrix.mtx")

# 读取细胞信息和基因信息
obs <- read.csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.obs.csv', row.names = 1) # 细胞数，细胞类型标签
var <- read.csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.var.csv', row.names = 1) # 基因数，基因名称

# 转置矩阵，因为Seurat期望的是基因在行、细胞在列
expression_matrix <- t(expression_matrix)
# Seurat内部用的是dgCMatrix（更常见、更快）
expression_matrix <- as(expression_matrix, "dgCMatrix")

# 确保基因信息设置为行名
rownames(expression_matrix) <- rownames(var)

# 确保样本信息设置为列名，这将连接表达矩阵的列与obs的行
colnames(expression_matrix) <- rownames(obs)

# 创建Seurat对象
seurat_object <- CreateSeuratObject(
  counts = expression_matrix,
  project = "NeuralOrigin",
  meta.data = obs
)

# seurat_object <- NormalizeData(seurat_object) # 本数据已经进行过normalized，不需要重复
# 旧版赋值方式：seurat_object[["RNA"]]@data <- expression_matrix  # 注意是标准化后的矩阵
# Seurat v5推荐赋值方式：把normalized表达矩阵写入data slot（v5写法）
seurat_object <- SetAssayData(
  object = seurat_object,
  # slot = "data",
  layer = "data",   # <--- 新用法
  new.data = expression_matrix
)
seurat_object <- FindVariableFeatures(seurat_object, selection.method = "vst", nfeatures = 2000)

umap <- read.csv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/Auco.OG.X_umap.csv", row.names=1)  # UMAP
umap <- umap[colnames(seurat_object), ]  # 对齐顺序
seurat_object[["umap"]] <- CreateDimReducObject(
  embeddings = as.matrix(umap),
  key = "UMAP_",
  assay = DefaultAssay(seurat_object)
)

dim(expression_matrix)
dim(obs)
dim(var)
seurat_object

# 保存Seurat对象为RDS文件
saveRDS(seurat_object, file = '/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/Auco.OG.normalized.rds')

In [ ]:
str(seurat_object)

In [ ]:
head(rownames(seurat_object))

In [ ]:
head(seurat_object@meta.data)

In [ ]:
DimPlot(seurat_object, reduction = "umap", group.by = "CellTypes")

In [ ]:
DimPlot(seurat_object, reduction = "umap", group.by = "Sub_cell_type")

In [ ]:
DimPlot(seurat_object, reduction = "umap", group.by = "seurat_clusters")

## 批量处理

In [ ]:
library(Seurat)
library(Matrix)

# 设置物种列表和路径
species <- c('Dare', 'Neve', 'Clhe', 'TrH1', 'TrH2', 'HoH13', 'ClH23', 'Spla')

indir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/mtx_obs_var_umap/"
outdir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/"

for (sp in species) {
  cat("Processing", sp, "...\n")
  prefix <- paste0(sp, ".OG")
  # 1. 读取表达矩阵
  expr_path <- file.path(indir, paste0(prefix, ".matrix.mtx"))
  obs_path  <- file.path(indir, paste0(prefix, ".obs.csv"))
  var_path  <- file.path(indir, paste0(prefix, ".var.csv"))
  umap_path <- file.path(indir, paste0(prefix, ".X_umap.csv"))
  
  expr <- readMM(expr_path)
  obs  <- read.csv(obs_path, row.names = 1)
  var  <- read.csv(var_path, row.names = 1)
  
  expr <- t(expr)
  # Seurat内部用的是dgCMatrix（更常见、更快）
  expr <- as(expr, "dgCMatrix")
  
  # 自动识别OG字段名
  # og_col <- if ("OrthoGene" %in% colnames(var)) "OrthoGene" else colnames(var)[1]
  # rownames(expr) <- var[[og_col]]
  rownames(expr) <- rownames(var)
  colnames(expr) <- rownames(obs)
  
  seu <- CreateSeuratObject(
    counts = expr,
    project = "NeuralOrigin",
    meta.data = obs
  )

  # seu <- NormalizeData(seu) # 本数据已经进行过normalized，不需要重复
  # 旧版赋值方式：seu[["RNA"]]@data <- expression_matrix  # 注意是标准化后的矩阵
  # Seurat v5推荐赋值方式：把normalized表达矩阵写入data slot（v5写法）
  seu <- SetAssayData(
    object = seu,
    # slot = "data",
    layer = "data",   # <--- 新用法
    new.data = expr
  )

  seu <- FindVariableFeatures(seu, selection.method = "vst", nfeatures = 2000)
  
  # 2. 添加UMAP（如有）
  if (file.exists(umap_path)) {
    umap <- read.csv(umap_path, row.names = 1)
    umap <- umap[colnames(seu), , drop=FALSE]  # 对齐顺序
    seu[["umap"]] <- CreateDimReducObject(
      embeddings = as.matrix(umap),
      key = "UMAP_",
      assay = DefaultAssay(seu)
    )
  }
  DimPlot(seu, reduction = "umap", group.by = "CellTypes")
  
  # 3. 保存RDS
  out_path <- file.path(outdir, paste0(prefix, ".normalized.rds"))
  saveRDS(seu, file = out_path)
  cat("Saved:", out_path, "\n")
}
